In [21]:
import os
import dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain.memory import ChatMessageHistory

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

llm=ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=40
)

了解即可 ConversationTokenBufferMemory

In [10]:
from langchain.memory import ConversationTokenBufferMemory
chat_history = ChatMessageHistory()
memory=ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=10,
    chat_memory=chat_history,
    return_messages=False,
    memory_key="history"
)

chat_history.add_message(HumanMessage(content="你好 我叫小明"))
chat_history.add_message(AIMessage(content="很高兴认识你"))
chat_history.add_message(HumanMessage(content="帮我回答一下1+2等于几"))
chat_history.add_message(AIMessage(content="3"))
chat_history.add_message(HumanMessage(content="一周有几天"))
chat_history.add_message(AIMessage(content="7"))
chat_history.add_message(HumanMessage(content="111"))
chat_history.add_message(AIMessage(content="111"))
chat_history.add_message(HumanMessage(content="222"))
chat_history.add_message(AIMessage(content="222"))
chat_history.add_message(HumanMessage(content="333"))
chat_history.add_message(AIMessage(content="333"))

print(memory.load_memory_variables({}))


{'history': 'Human: 你好 我叫小明\nAI: 很高兴认识你\nHuman: 帮我回答一下1+2等于几\nAI: 3\nHuman: 一周有几天\nAI: 7\nHuman: 111\nAI: 111\nHuman: 222\nAI: 222\nHuman: 333\nAI: 333'}


ConversationSummaryMemory

In [13]:
# 1. 导入相关包（补充消息角色类）
from langchain.memory import ConversationSummaryMemory, ChatMessageHistory

# 2. 创建大模型
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 3. 初始化消息存储 & 总结记忆（参数不变）
chat_history = ChatMessageHistory()
memory = ConversationSummaryMemory(
    llm=llm,
    chat_memory=chat_history,
    return_messages=False,
    memory_key="history"
)

# 4. 存储消息（关键修正：用 add_message 传入消息对象，再用 save_context 触发总结）
# 第1轮对话
human_msg1 = HumanMessage(content="你好")
ai_msg1 = AIMessage(content="怎么了")
chat_history.add_message(human_msg1)
chat_history.add_message(ai_msg1)
# 调用 save_context，让记忆模块记录并参与后续总结
memory.save_context(inputs={"input": human_msg1.content}, outputs={"output": ai_msg1.content})

# 第2轮对话
human_msg2 = HumanMessage(content="你是谁")
ai_msg2 = AIMessage(content="我是AI助手小智")
chat_history.add_message(human_msg2)
chat_history.add_message(ai_msg2)
memory.save_context(inputs={"input": human_msg2.content}, outputs={"output": ai_msg2.content})

# 第3轮对话
human_msg3 = HumanMessage(content="初次对话，你能介绍一下你自己吗？")
ai_msg3 = AIMessage(content="当然可以了。我是一个无所不能的小智。")
chat_history.add_message(human_msg3)
chat_history.add_message(ai_msg3)
memory.save_context(inputs={"input": human_msg3.content}, outputs={"output": ai_msg3.content})

# 5. 读取消息（获取总结后的对话历史）
summary_result = memory.load_memory_variables({})
print("对话总结结果：")
print(summary_result)

对话总结结果：
{'history': 'The human greets the AI with "你好" (hello), and the AI responds by asking, "怎么了" (what\'s wrong). The human then asks, "你是谁" (who are you), and the AI replies that it is "AI助手小智" (AI assistant Xiao Zhi). The human requests an introduction, and the AI describes itself as "无所不能的小智" (an all-powerful Xiao Zhi).'}


In [20]:
# 1. 初始化对话历史
chat_history = ChatMessageHistory()
chat_history.add_user_message("你好，你是谁？")
chat_history.add_ai_message("我是AI助手")

# 2. 初始化LLM和总结记忆
llm = ChatOpenAI(model="gpt-4o-mini")
memory = ConversationSummaryMemory.from_messages(
    llm=llm,
    #是生成摘要的原材料 保留完整对话供必要时回溯
    chat_memory=chat_history,
    return_messages=True
)

# 3. 新增对话（关键：用 save_context 记录，触发总结更新）
new_human_msg = "帮我回答一下1+2等于几"
new_ai_msg = "3"
# 主动调用 save_context，让记忆模块纳入新对话并更新总结
memory.save_context(inputs={"input": new_human_msg}, outputs={"output": new_ai_msg})

# 4. 读取更新后的总结
result = memory.load_memory_variables({})
print(memory.chat_memory.messages)
print('\n')
print(result)

[HumanMessage(content='你好，你是谁？', additional_kwargs={}, response_metadata={}), AIMessage(content='我是AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我回答一下1+2等于几', additional_kwargs={}, response_metadata={}), AIMessage(content='3', additional_kwargs={}, response_metadata={})]


{'history': [SystemMessage(content='The human greets and asks who the AI is. The AI responds that it is an AI assistant. The human then asks the AI what 1 + 2 equals, and the AI answers that it is 3.', additional_kwargs={}, response_metadata={})]}
